
# GRAHSP SFH figure reproduction: delayed-tau galaxy SED sweep

Reproduction of the star-formation-history overview figure of Buchner et al.
(2024, GRAHSP): galaxy SEDs for a delayed-$\tau$ SFH
($\mathrm{SFR}\propto t\,e^{-t/\tau}$, CIGALE ``sfh_delayed``) whose
cutoff timescale $\tau$ is swept from 100 Myr (yellow; SFR truncates
early, old-star-dominated) to 10 Gyr (dark blue; continuously rising, young,
nebular- and dust-rich). Minimal attenuation E(B-V)=0.01 is applied. The inset
shows the corresponding star-formation histories.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import warnings

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm, colors

import tengri
from tengri import FIXED, Fixed, SEDModel
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

C_NM_HZ = 2.99792458e17
ERG_TO_W = 1e-7

# No-arg load_ssp() auto-discovers the bundled PARSEC/MILES/Chabrier wNE grid
# regardless of the working directory (sphinx-gallery executes from elsewhere).
ssp = tengri.load_ssp()
wave_aa = jnp.logspace(np.log10(300.0), np.log10(1.0e8), 3500)  # 0.03 um - 1e4 um
wave_um = np.asarray(wave_aa) / 1e4

AGE_GYR = 4.0
tau_grid_myr = np.array([100.0, 300.0, 1000.0, 3000.0, 10000.0])
norm = colors.LogNorm(vmin=100.0, vmax=10000.0)
cmap = cm.get_cmap("cividis_r")

fig, ax = plt.subplots(figsize=(7.4, 6.0))
axin = ax.inset_axes([0.13, 0.13, 0.40, 0.30])

# Build the model ONCE; the delayed-tau SFH timescale ``sfh_delayed_tau_gyr``
# is a parameter, so the sweep just overrides that key per iteration and re-runs
# the (already-compiled) forward pass — both predict_rest_sed and predict_sfh
# read it from the params dict. Rebuilding inside the loop would recompile the
# SSP pipeline every iteration and accumulate XLA buffers (gallery-OOM).
model = SEDModel.build(
    ssp_data=ssp,
    sfh={
        "type": "delayed",
        "*": FIXED,
        "tau_gyr": tau_grid_myr[0] / 1e3,  # baseline; overridden per tau below
        "age_gyr": AGE_GYR,
        "log_total_mass": 10.5,
    },
    dust={
        "type": "two_component",
        "law_bc": "calzetti",
        "*": FIXED,
        "tau_bc": 0.03,
        "tau_diff": 0.01,
        "emission": {"type": "dale2014", "*": FIXED},
    },
    redshift=Fixed(0.01),
)
base_params = model.spec.get_fixed_values()

for tau_myr in tau_grid_myr:
    params = {**base_params, "sfh_delayed_tau_gyr": tau_myr / 1e3}
    rest = model.predict_rest_sed(params, wave_aa)  # (wave, L_nu) erg/s/Hz
    lnu = np.asarray(rest[1] if isinstance(rest, tuple) or np.ndim(rest) == 2 else rest)
    lam_Llam_W = lnu * (C_NM_HZ / (np.asarray(wave_aa) * 0.1)) * ERG_TO_W
    ax.plot(wave_um, lam_Llam_W, color=cmap(norm(tau_myr)), lw=1.3, zorder=3)

    sfh = model.predict_sfh(params)
    t_gyr = np.asarray(sfh["t_gyr"])  # lookback time: 0 = present, AGE_GYR = formation
    sfr = np.asarray(sfh["sfr_full"])
    # Convert to time-since-formation (present at right, as in the paper) and
    # normalize to the peak ("a.u.") so each tau's shape is visible.
    t_since = (AGE_GYR - t_gyr) * 1e3  # Myr
    keep = (t_since >= 0) & (t_gyr <= AGE_GYR)
    axin.plot(t_since[keep], sfr[keep] / sfr.max(), color=cmap(norm(tau_myr)), lw=1.3)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.03, 1e4)
ax.set_ylim(1e30, 1e39)
ax.set_xlabel(r"Wavelength [$\mu$m]")
ax.set_ylabel(r"Luminosity $\lambda L_\lambda$ [W]")

axin.set_xlim(0, AGE_GYR * 1e3)
axin.set_title("SFH", fontsize=9)
axin.set_xlabel("t [Myr]", fontsize=8)
axin.set_ylabel("SFR [a.u.]", fontsize=8)
axin.tick_params(labelsize=7)
axin.set_yticks([])

secax = ax.secondary_xaxis(
    "top", functions=(lambda x: C_NM_HZ / 1e3 / x, lambda nu: C_NM_HZ / 1e3 / nu)
)
secax.set_xlabel("Frequency [Hz]")

sm = cm.ScalarMappable(norm=norm, cmap=cmap)
cbar = fig.colorbar(sm, ax=ax, fraction=0.05, pad=0.02)
cbar.set_label(r"$\tau$ [Myr]")

fig.tight_layout()
plt.savefig("plot_grahsp_paper_sfh_tau_sweep.png", dpi=150, bbox_inches="tight")